In [10]:
import torch
import torchvision
import torchvision.transforms as transforms
import time

# -----------------------------
# Load Dataset (same as before)
# -----------------------------
transform = transforms.ToTensor()

train_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)

test_dataset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

In [11]:
# Faster subset
subset = torch.utils.data.Subset(train_dataset, range(5000))

train_loader = torch.utils.data.DataLoader(subset, batch_size=128, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64)

In [12]:
# -----------------------------
# Model Parameters (UPDATED)
# -----------------------------
input_size = 784
h1_size = 256   # increased
h2_size = 128   # increased
output_size = 10

W1 = torch.randn(input_size, h1_size, requires_grad=True)
b1 = torch.zeros(h1_size, requires_grad=True)

W2 = torch.randn(h1_size, h2_size, requires_grad=True)
b2 = torch.zeros(h2_size, requires_grad=True)

W3 = torch.randn(h2_size, output_size, requires_grad=True)
b3 = torch.zeros(output_size, requires_grad=True)

lr = 0.01
epochs = 5

In [ ]:
import time

# -----------------------------
# Training
# -----------------------------
start_time = time.time()

for epoch in range(epochs):
    total_loss = 0
    total_samples = 0   # track number of samples

    for images, labels in train_loader:

        x = images.reshape(-1, 784)

        # Forward
        z1 = x @ W1 + b1
        h1 = torch.relu(z1)

        z2 = h1 @ W2 + b2
        h2 = torch.relu(z2)

        logits = h2 @ W3 + b3

        # Loss
        loss = torch.nn.functional.cross_entropy(logits, labels)

        # Backward
        loss.backward()

        # Update
        with torch.no_grad():
            W1 -= lr * W1.grad
            b1 -= lr * b1.grad

            W2 -= lr * W2.grad
            b2 -= lr * b2.grad

            W3 -= lr * W3.grad
            b3 -= lr * b3.grad

        # Zero grad
        W1.grad.zero_()
        b1.grad.zero_()
        W2.grad.zero_()
        b2.grad.zero_()
        W3.grad.zero_()
        b3.grad.zero_()

        #  Proper accumulation (per sample)
        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

    #  True average loss
    avg_loss = total_loss / total_samples

    print(f"Epoch {epoch+1}, Avg Loss: {avg_loss:.4f}")

end_time = time.time()
training_time = end_time - start_time

print(f"Training Time: {training_time:.2f} seconds")

Epoch 1, Avg Loss: 319.3219
Epoch 2, Avg Loss: 21.3919
Epoch 3, Avg Loss: 12.9228
Epoch 4, Avg Loss: 9.3296
Epoch 5, Avg Loss: 6.7076
Training Time: 6.35 seconds


In [15]:
# -----------------------------
# Evaluation (Accuracy)
# -----------------------------
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:

        x = images.reshape(-1, 784)

        z1 = x @ W1 + b1
        h1 = torch.relu(z1)

        z2 = h1 @ W2 + b2
        h2 = torch.relu(z2)

        logits = h2 @ W3 + b3

        preds = torch.argmax(logits, dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total


# -----------------------------
# Final Results
# -----------------------------
print(f"\nFinal Accuracy: {accuracy:.2f}%")
print(f"Training Time: {training_time:.2f} seconds")


Final Accuracy: 57.65%
Training Time: 6.35 seconds


In [ ]:
# Comparison: Part 3 vs Part 4
# Metric	Part 3 (Small Model: 32→16)	Part 4 (Large Model: 256→128)
# Initial Loss	30.45	319.32
# Final Loss	2.60	6.70
# Test Accuracy	40.99%	57.65%
# Training Time	~5.8 sec	~6.35 sec

# Observations
# 1. Loss Behavior
# Part 4 starts with much higher loss
# Reason: larger network → larger random weights → unstable start
# But it reduces consistently, showing proper learning
# 2. Accuracy Improvement
# Part 3: 40.99%
# Part 4: 57.65%

#  Significant improvement (~17%)
# This confirms:
# Larger network learns better feature representations
# 3. Training Time
# Slight increase (~0.5 sec more)

#  Meaning:
# Performance gain is achieved with only minor extra cost
# 4. Learning Capacity
# Small model → limited ability (underfitting)
# Large model → better pattern capture

# Critical Insight (important for lab)

# Even though:
# Part 4 has higher final loss (6.70 vs 2.60)
# It still gives better accuracy

# This shows:
# Loss value alone is not enough — accuracy is the real performance metric

# The larger fully connected network (256 → 128) performed better than the smaller network (32 → 16).
# Although the larger model started with a higher initial loss due to increased parameter size, 
# it achieved significantly higher test accuracy (57.65% vs 40.99%). 
# The training time increased slightly but remained comparable. 
# This demonstrates that increasing model capacity improves learning and classification performance, 
# though at a small computational cost.